### We'll see two approaches below to parse the docs into a format that RNN Model can understand.

In [2]:
import numpy as np

docs = ['go india',
        'india india',
        'hip hip hurray',
        'jeetega bhai jeetega india jeetega',
        'bharat mata ki jai',
        'kohli kohli',
        'sachin sachin',
        'dhoni dhoni',
        'modi ji ki jai',
        'inquilab zindabad']

In [3]:
docs

['go india',
 'india india',
 'hip hip hurray',
 'jeetega bhai jeetega india jeetega',
 'bharat mata ki jai',
 'kohli kohli',
 'sachin sachin',
 'dhoni dhoni',
 'modi ji ki jai',
 'inquilab zindabad']

### Approach I: Integer Indexing of Terms

In [8]:
from keras.preprocessing.text import Tokenizer
tk = Tokenizer(oov_token='<OOV>')

In [11]:
tk.fit_on_texts(docs)
tk.word_index

{'<OOV>': 1,
 'india': 2,
 'jeetega': 3,
 'hip': 4,
 'ki': 5,
 'jai': 6,
 'kohli': 7,
 'sachin': 8,
 'dhoni': 9,
 'go': 10,
 'hurray': 11,
 'bhai': 12,
 'bharat': 13,
 'mata': 14,
 'modi': 15,
 'ji': 16,
 'inquilab': 17,
 'zindabad': 18}

In [16]:
seq = tk.texts_to_sequences(docs)
seq

[[10, 2],
 [2, 2],
 [4, 4, 11],
 [3, 12, 3, 2, 3],
 [13, 14, 5, 6],
 [7, 7],
 [8, 8],
 [9, 9],
 [15, 16, 5, 6],
 [17, 18]]

In [25]:
from keras.utils import pad_sequences
padded_seq = pad_sequences(seq, padding='post')
padded_seq

array([[10,  2,  0,  0,  0],
       [ 2,  2,  0,  0,  0],
       [ 4,  4, 11,  0,  0],
       [ 3, 12,  3,  2,  3],
       [13, 14,  5,  6,  0],
       [ 7,  7,  0,  0,  0],
       [ 8,  8,  0,  0,  0],
       [ 9,  9,  0,  0,  0],
       [15, 16,  5,  6,  0],
       [17, 18,  0,  0,  0]])

In [26]:
from keras.datasets import imdb
from keras import Sequential
from keras.layers import Dense, SimpleRNN, Embedding, Flatten

In [27]:
(X_train, y_train), (X_test, y_test) = imdb.load_data()

17464789/17464789 [==============================] - 4s 0us/step


In [56]:
X_train = pad_sequences(X_train, padding='post', maxlen=150)
X_test  = pad_sequences(X_test,  padding='post', maxlen=150)

In [57]:
X_train[0].shape

(150,)

In [ ]:
model = Sequential()
# here input shape is (150, 1) because we have 150 words in each review and each word is represented by an integer
# return_sequences is False because we want to get the output of the last final time-step only, if 
model.add(SimpleRNN(32, input_shape=(150, 1), return_sequences=False))
model.add(Dense(1, activation='sigmoid'))

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 simple_rnn (SimpleRNN)      (None, 32)                1088      
                                                                 
 dense (Dense)               (None, 1)                 33        
                                                                 
Total params: 1,121
Trainable params: 1,121
Non-trainable params: 0
_________________________________________________________________


In [59]:
model.compile(optimizer='adam', loss='binary_crossentropy',
              metrics=['accuracy'])
model.fit(X_train, y_train,
          epochs=5, validation_data=(X_test, y_test))

Epoch 1/5
782/782 [==============================] - 152s 191ms/step - loss: 0.6934 - accuracy: 0.5004 - val_loss: 0.6932 - val_accuracy: 0.5000
Epoch 2/5
782/782 [==============================] - 143s 183ms/step - loss: 0.6933 - accuracy: 0.5005 - val_loss: 0.6932 - val_accuracy: 0.5000
Epoch 3/5
782/782 [==============================] - 152s 194ms/step - loss: 0.6940 - accuracy: 0.5037 - val_loss: 0.6936 - val_accuracy: 0.5000
Epoch 4/5
782/782 [==============================] - 154s 197ms/step - loss: 0.6941 - accuracy: 0.4912 - val_loss: 0.6932 - val_accuracy: 0.5000
Epoch 5/5
782/782 [==============================] - 154s 197ms/step - loss: 0.6934 - accuracy: 0.4968 - val_loss: 0.6932 - val_accuracy: 0.5000


## Approach II: Embeddings

In [64]:
tk.word_index

{'<OOV>': 1,
 'india': 2,
 'jeetega': 3,
 'hip': 4,
 'ki': 5,
 'jai': 6,
 'kohli': 7,
 'sachin': 8,
 'dhoni': 9,
 'go': 10,
 'hurray': 11,
 'bhai': 12,
 'bharat': 13,
 'mata': 14,
 'modi': 15,
 'ji': 16,
 'inquilab': 17,
 'zindabad': 18}

In [60]:
seq

[[10, 2],
 [2, 2],
 [4, 4, 11],
 [3, 12, 3, 2, 3],
 [13, 14, 5, 6],
 [7, 7],
 [8, 8],
 [9, 9],
 [15, 16, 5, 6],
 [17, 18]]

In [76]:
padded_seq

array([[10,  2,  0,  0,  0],
       [ 2,  2,  0,  0,  0],
       [ 4,  4, 11,  0,  0],
       [ 3, 12,  3,  2,  3],
       [13, 14,  5,  6,  0],
       [ 7,  7,  0,  0,  0],
       [ 8,  8,  0,  0,  0],
       [ 9,  9,  0,  0,  0],
       [15, 16,  5,  6,  0],
       [17, 18,  0,  0,  0]])

In [ ]:
model_emb = Sequential()
# here input_dim is 18 because we have 18 unique words in our vocabulary, 
# output_dim is 2 because we want to convert each integer into a vector of size 2, 
# and input_length is 5 because we have padded our sequences to a length of 5.
model_emb.add(Embedding(input_dim=18, output_dim=2, input_length=5))
model_emb.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_2 (Embedding)     (None, 5, 2)              36        
                                                                 
Total params: 36
Trainable params: 36
Non-trainable params: 0
_________________________________________________________________


In [81]:
model_emb.compile(optimizer='adam',
              metrics=['accuracy'])

In [ ]:
preds = model_emb.predict(padded_seq)
preds

1/1 [==============================] - 0s 119ms/step


array([[[ 1.56462193e-07,  3.78343128e-02],
        [ 2.04745866e-02, -3.56756151e-04],
        [ 3.72077934e-02,  1.20000616e-02],
        [ 3.72077934e-02,  1.20000616e-02],
        [ 3.72077934e-02,  1.20000616e-02]],

       [[ 2.04745866e-02, -3.56756151e-04],
        [ 2.04745866e-02, -3.56756151e-04],
        [ 3.72077934e-02,  1.20000616e-02],
        [ 3.72077934e-02,  1.20000616e-02],
        [ 3.72077934e-02,  1.20000616e-02]],

       [[ 3.99952866e-02, -9.46285576e-03],
        [ 3.99952866e-02, -9.46285576e-03],
        [-4.64868546e-03, -3.32057029e-02],
        [ 3.72077934e-02,  1.20000616e-02],
        [ 3.72077934e-02,  1.20000616e-02]],

       [[ 6.64277002e-03,  4.05063517e-02],
        [-1.26351230e-02, -6.63302839e-04],
        [ 6.64277002e-03,  4.05063517e-02],
        [ 2.04745866e-02, -3.56756151e-04],
        [ 6.64277002e-03,  4.05063517e-02]],

       [[ 2.97595896e-02,  3.39118950e-02],
        [ 1.50926225e-02, -3.24716344e-02],
        [-3.12263016e-02

In [86]:
X_train.shape

(25000, 150)

In [89]:
# Embedding layer is a lookup table which maps each integer to a vector of fixed size. 
# It is used to convert the integer encoded sequences into dense vectors of fixed size. 

# The input_dim is the size of the vocabulary, 
# output_dim is the size of the embedding vector, 
# and input_length is the length of the input sequences.

emb_model = Sequential()
emb_model.add(Embedding(input_dim=10000, output_dim=2, input_length=150))
emb_model.add(SimpleRNN(32, return_sequences=False)) 
emb_model.add(Dense(1, activation='sigmoid'))
emb_model.summary()

Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_4 (Embedding)     (None, 150, 2)            20000     
                                                                 
 simple_rnn_2 (SimpleRNN)    (None, 32)                1120      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 21,153
Trainable params: 21,153
Non-trainable params: 0
_________________________________________________________________


In [91]:
emb_model.compile(optimizer='adam', loss='binary_crossentropy',
              metrics=['accuracy'])
emb_model.fit(X_train, y_train,
          epochs=5, validation_data=(X_test, y_test))

Epoch 1/5
782/782 [==============================] - 234s 298ms/step - loss: 0.6934 - accuracy: 0.5010 - val_loss: 0.6932 - val_accuracy: 0.5000
Epoch 2/5
782/782 [==============================] - 206s 264ms/step - loss: 0.6933 - accuracy: 0.4970 - val_loss: 0.6932 - val_accuracy: 0.5000
Epoch 3/5
782/782 [==============================] - 206s 263ms/step - loss: 0.6933 - accuracy: 0.4982 - val_loss: 0.6932 - val_accuracy: 0.5000
Epoch 4/5
782/782 [==============================] - 206s 263ms/step - loss: 0.6932 - accuracy: 0.5004 - val_loss: 0.6933 - val_accuracy: 0.5000
Epoch 5/5
782/782 [==============================] - 205s 262ms/step - loss: 0.6933 - accuracy: 0.4990 - val_loss: 0.6932 - val_accuracy: 0.5000
